Tiktokenizer - https://tiktokenizer.vercel.app/?model=gpt2

here you visualize tokenization of text


<img src="public/tiktokenizer.png" width="500">

In [274]:
# download text in file
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    file_path = "the-verdict.txt"
    response = requests.get(url)
    with open(file_path, "wb") as f:
        f.write(response.content)

In [275]:
# load text
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [276]:
len(raw_text)

20479

In [277]:
print(raw_text[:20])
print("...")
print(raw_text[-20:])

I HAD always thought
...
ng our kind of art."


In [278]:
# split text into tokens
import re

text = "Hello, world. This, is a test."
result = re.split(r"(\s)", text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [279]:
# comma and period as separate token
result = re.split(r"([,.]|\s)", text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [280]:
# remove whitespaces
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


In [281]:
# more symbols as separate tokens
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [282]:
# raw_text into tokens
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
preprocesssed = result

In [283]:
print(preprocesssed[:10])
print("...")
print(preprocesssed[-10:])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius']
...
["'", 's', 'no', 'exterminating', 'our', 'kind', 'of', 'art', '.', '"']


In [284]:
len(preprocesssed)

4690

In [285]:
all_words = sorted(set(preprocesssed))
print(all_words[:10])
print("...")
print(all_words[-10:])

['!', '"', "'", '(', ')', ',', '--', '.', ':', ';']
...
['would', 'wouldn', 'year', 'years', 'yellow', 'yet', 'you', 'younger', 'your', 'yourself']


In [286]:
vocab_size = len(all_words)
vocab_size

1130

In [287]:
vocab = {token: integer for integer, token in enumerate(all_words)}

In [288]:
items = list(vocab.items())
print(items[:5])
print("...")
print(items[-5:])

[('!', 0), ('"', 1), ("'", 2), ('(', 3), (')', 4)]
...
[('yet', 1125), ('you', 1126), ('younger', 1127), ('your', 1128), ('yourself', 1129)]


In [289]:
from typing import Any


class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return text

In [290]:
tokenizer = SimpleTokenizerV1(vocab)
tokenizer.decode(tokenizer.encode("What are you doing?"))

'What are you doing ?'

Note that in above example there's a space before question mark which is undesired

In [291]:
# Replace spaces before the specified punctuations
class SimpleTokenizerV1:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r"\1", text)
        return text

In [292]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [293]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [294]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [295]:
tokenizer.decode(tokenizer.encode("What are you doing?"))

'What are you doing?'

In [296]:
# note that tokenizer will only work for words present in raw_text based on which vocabulary is built
try:
    text1 = "Hello, do you like tea?"
    ids = tokenizer.encode(text1)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

KeyError: 'Hello'


In [297]:
# adding special tokens
# handling unknown vocab (though this method leads to loss of information)
# it's better to split unknown words into character tokens
all_tokens = sorted(set(preprocesssed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token: integer for integer, token in enumerate(all_tokens)}

In [298]:
len(vocab.items())

1132

In [299]:
for item in list(vocab.items())[-5:]:
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [300]:
class SimpleTokenizerV2:
    def __init__(self, vocab: dict[str | Any, int]):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r"\1", text)
        return text

In [301]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [302]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [303]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

In [304]:
import tiktoken

tiktoken.__version__

'0.12.0'

In [305]:
tokenizer = tiktoken.get_encoding("gpt2")

In [306]:
text1 = "Hello, do you like tea?"
tokenizer.encode(text1)

[15496, 11, 466, 345, 588, 8887, 30]

In [307]:
tokenizer.decode(tokenizer.encode(text1))

'Hello, do you like tea?'

In [308]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."
)
try:
    tokenizer.encode(text)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

ValueError: Encountered text corresponding to disallowed special token '<|endoftext|>'.
If you want this text to be encoded as a special token, pass it to `allowed_special`, e.g. `allowed_special={'<|endoftext|>', ...}`.
If you want this text to be encoded as normal text, disable the check for this token by passing `disallowed_special=(enc.special_tokens_set - {'<|endoftext|>'})`.
To disable this check for all special tokens, pass `disallowed_special=()`.



In [309]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


<img src="public/tiktokenizer-example-1.png" width="500"/>

In [310]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [311]:
tokenizer.decode(tokenizer.encode(text, disallowed_special=()))

'Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.'

In [312]:
ids = tokenizer.encode("My name is Rahul Gupta, with some random text")
for id in ids:
    print(f'{id} -> "{tokenizer.decode([id])}"')

3666 -> "My"
1438 -> " name"
318 -> " is"
48543 -> " Rahul"
42095 -> " Gupta"
11 -> ","
351 -> " with"
617 -> " some"
4738 -> " random"
2420 -> " text"


In [313]:
ids = tokenizer.encode("My name is somerandomjijiffdafds jijutsu")
for id in ids:
    print(f'{id} -> "{tokenizer.decode([id])}"')

3666 -> "My"
1438 -> " name"
318 -> " is"
3870 -> " som"
263 -> "er"
3749 -> "andom"
73 -> "j"
2926 -> "ij"
733 -> "iff"
67 -> "d"
1878 -> "af"
9310 -> "ds"
474 -> " j"
2926 -> "ij"
36567 -> "utsu"


<img src="public/tiktokenizer-random-chars.png" width="500"/>

In [314]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [315]:
for id in enc_text[:10]:
    print(f'{id} -> "{tokenizer.decode([id])}"')

40 -> "I"
367 -> " H"
2885 -> "AD"
1464 -> " always"
1807 -> " thought"
3619 -> " Jack"
402 -> " G"
271 -> "is"
10899 -> "burn"
2138 -> " rather"


In [316]:
enc_sample = enc_text

In [317]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1 : context_size + 1]

print(f"x: {x}")
print(f"y:     {y}")

x: [40, 367, 2885, 1464]
y:     [367, 2885, 1464, 1807]


In [318]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[40] ----> 367
[40, 367] ----> 2885
[40, 367, 2885] ----> 1464
[40, 367, 2885, 1464] ----> 1807


In [319]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

I ---->  H
I H ----> AD
I HAD ---->  always
I HAD always ---->  thought


In [320]:
import torch

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu130


In [321]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, (
            "Number of tokenized inputs must at least be equal to max_length+1"
        )

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]
            target_chunk = token_ids[i + 1 : i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [322]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )

    return dataloader

In [323]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [324]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [325]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [326]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=2, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [327]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[2885, 1464, 1807, 3619]]), tensor([[1464, 1807, 3619,  402]])]


In [328]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [329]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1807, 3619,  402,  271]]), tensor([[ 3619,   402,   271, 10899]])]


In [330]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
